# 01 — Integrate official PriceCatcher records

Run notebooks **01 → 06**. Settings live in `src/pricecatcher_config.py`.
This notebook freezes lookup metadata and the food basket, caches monthly source files,
standardises egg units, and saves a combined Parquet plus a SHA-256 source manifest.

**Important:** the original shared configuration was not supplied. The replacement basket uses
27 explicit official item codes, not a claim to reproduce the original basket exactly. Supply a root-level
`basket.csv` (`item_code,project_group`) before the first run to preserve an exact basket.
Inspect the exported basket. Different item codes remain separate even after unit conversion.
Current lookup snapshots do not establish historical shop metadata.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Run from the repository root or its notebooks folder.")
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
import numpy as np
import pricecatcher_config as cfg
from data_utils import save_json, surge_event
from datetime import datetime, timezone
from data_utils import fetch_file, file_hash, select_basket, normalise_units

raw_dir = cfg.DATA_DIR / "raw"
lookup_dir = cfg.DATA_DIR / "lookups"
output = cfg.stage(1)
items_path, premises_path = lookup_dir / "lookup_item.parquet", lookup_dir / "lookup_premise.parquet"
fetch_file(cfg.source_file("lookup_item.parquet"), items_path)
fetch_file(cfg.source_file("lookup_premise.parquet"), premises_path)
items, premises = pd.read_parquet(items_path), pd.read_parquet(premises_path)
for table, key in [(items, "item_code"), (premises, "premise_code")]:
    table[key] = pd.to_numeric(table[key], errors="raise").astype("int64")
    if table[key].isna().any() or table[key].duplicated().any():
        raise ValueError(f"Lookup needs unique non-missing {key}.")

basket_path = lookup_dir / "basket.csv"
expected_basket = select_basket(items)
if not basket_path.exists():
    expected_basket.to_csv(basket_path, index=False)
basket = pd.read_csv(basket_path)
if basket.empty or basket["item_code"].duplicated().any():
    raise ValueError("Frozen basket is empty or has duplicate codes.")
expected_codes = expected_basket[["item_code", "project_group"]].sort_values("item_code").reset_index(drop=True)
frozen_codes = basket[["item_code", "project_group"]].sort_values("item_code").reset_index(drop=True)
if not expected_codes.equals(frozen_codes):
    raise ValueError("Basket settings differ from the frozen basket. Use new data/output directories.")
basket.to_csv(output / "basket.csv", index=False)
display(basket[["item_code", "item", "unit", "project_group"]])


,item_code,item,unit,project_group
0,1,AYAM BERSIH - STANDARD,1kg,Chicken
1,2,AYAM BERSIH - SUPER,1kg,Chicken
2,118,TELUR AYAM GRED A (BERAT 65.0 GM HINGGA 69.9 G...,10 biji,Eggs
3,119,TELUR AYAM GRED B (BERAT 60.0 GM HINGGA 64.9 GM),10 biji,Eggs
4,120,TELUR AYAM GRED C (BERAT 55.0 GM HINGGA 59.9 G...,10 biji,Eggs
5,1109,TELUR AYAM GRED A,30 biji,Eggs
6,1110,TELUR AYAM GRED B,30 biji,Eggs
7,1111,TELUR AYAM GRED C,30 biji,Eggs
8,992,BERAS CAP JATI (SST5%),10 kg,Rice
9,1581,BERAS CAP RAMBUTAN (SST5%),10 kg,Rice


## Integrate each month
Conflicting or duplicate date–premise–item keys stop execution. Missing joins stop execution
rather than silently removing remote districts. Invalid numeric prices are retained here for
Notebook 03 to count and remove. Failed downloads never export a partial combined dataset.

In [2]:
frames, manifest_rows = [], []
item_columns = ["item_code", "item", "unit", "project_group"]
premise_columns = ["premise_code", "premise", "state", "district", "premise_type"]
for month in pd.period_range(cfg.START_MONTH, cfg.END_MONTH, freq="M").astype(str):
    filename = f"pricecatcher_{month}.parquet"
    path = raw_dir / filename
    fetch_file(cfg.source_file(filename), path)
    data = pd.read_parquet(path, columns=["date", "premise_code", "item_code", "price"])
    data["item_code"] = pd.to_numeric(data["item_code"], errors="raise").astype("int64")
    data = data.loc[data["item_code"].isin(basket["item_code"])].copy()
    data["premise_code"] = pd.to_numeric(data["premise_code"], errors="raise").astype("int64")
    data["date"] = pd.to_datetime(data["date"], errors="raise").dt.normalize()
    data["price"] = pd.to_numeric(data["price"], errors="coerce")
    if data.empty or not data["date"].dt.to_period("M").eq(pd.Period(month)).all():
        raise ValueError(f"{month}: empty selected data, missing dates, or wrong month.")
    data = data.merge(basket[item_columns], on="item_code", how="left", validate="many_to_one")
    data = data.merge(premises[premise_columns], on="premise_code", how="left", validate="many_to_one")
    text_columns = ["item", "unit", "project_group", "premise", "state", "district", "premise_type"]
    for column in text_columns:
        data[column] = data[column].astype("string").str.strip().replace("", pd.NA)
    if data[text_columns].isna().any().any():
        raise ValueError(f"{month}: missing lookup details. Inspect joins before continuing.")
    if data.duplicated(["date", "premise_code", "item_code"]).any():
        raise ValueError(f"{month}: duplicate date–premise–item keys need source review.")
    data = normalise_units(data)
    frames.append(data)
    manifest_rows.append({"month": month, "selected_rows": len(data), "source": cfg.source_file(filename),
                          "sha256": file_hash(path), "first_date": str(data["date"].min().date()),
                          "last_date": str(data["date"].max().date())})
    print(month, len(data))

combined = pd.concat(frames, ignore_index=True)
del frames
combined.to_parquet(output / "integrated.parquet", index=False)
save_json(output / "source_manifest.json", {
    "created_utc": datetime.now(timezone.utc).isoformat(), "rows": len(combined),
    "source_mode": "local mirror (verify provenance)" if cfg.SOURCE_DIR else "official HTTPS",
    "lookup_item_sha256": file_hash(items_path), "lookup_premise_sha256": file_hash(premises_path),
    "basket_sha256": file_hash(basket_path), "months": manifest_rows,
    "egg_rule": "30 biji divided by 3; 10 biji unchanged; item codes remain separate",
})
print("Saved", len(combined), "records to", output)


2025-01 281259
2025-02 244006
2025-03 289506
2025-04 271926
2025-05 302496
2025-06 262105
2025-07 332641
2025-08 358949
2025-09 302817
2025-10 350767
2025-11 339399
2025-12 340599
2026-01 357752
2026-02 266557
2026-03 284682
2026-04 351381
2026-05 326147
2026-06 318623
Saved 5581612 records to /Users/alfredyap/Documents/GitHub/Food/outputs/01
